### Here we demodulate all scans from the same observation of the calsource in order to recreate the maps of the synthbeam. This is the "all scans" version of `Scan_Demodulation_Src.ipynb`.

Some adjustments have to be done on the 2019-04 datasets:
- the three datasets (data, hk and calsource) should have the same t0, so need to synch them with t_xxx = t_xxx + (t_hk_0 - t_xxx_0)

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# Allows the figures to be dynamic
%matplotlib ipympl

In [ ]:
from matplotlib import rc
rc('figure',figsize=(16,8))
rc('font',size=12)
rc('text',usetex=False)

from qubicpack.qubicfp import qubicfp
import qubic.lib.Calibration.Qfiber as ft

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import scipy.ndimage as ndimg
import glob
import string

import iminuit
from iminuit.cost import LeastSquares
import fitsio
from astropy.time import Time
import scipy.signal as scsig
import healpy as hp

import fitsio

from fast_histogram import histogram2d
from scipy.interpolate import RegularGridInterpolator

In [ ]:
rc('figure',figsize=(10,7))

## Dataset

Get the directories corresponding to the days we consider:

In [ ]:
# days = ['2019-04-18']
days = ['2019-04-17', '2019-04-18']
# data_dir = '/qubic/Data/Calib-TD/'+day+'/'
data_dir = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_TOD/"
data_dir_calsrc = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_TOD/calsource/"
print(data_dir)
names = "ScanMap_170GHz_Speed_VE4_El"
# names = "ScanMap_170GHz_Speed_VE4_El_52.81"
dirs = []
fcalsrc = []
for day in days:
    dirs.append(np.sort(glob.glob(data_dir + day + '/*{}*'.format(names))))
    ##### The date format for calsrc has no '-' so we need to change the day to this very format.
    daycalsrc = "".join(str.split(day,'-'))
    fcalsrc.append(np.sort(glob.glob(data_dir_calsrc + "*" + daycalsrc + "*")))
if len(days) > 1: # isinstance(my_array, np.ndarray)
    dirs = np.concatenate(dirs)
    fcalsrc = np.concatenate(fcalsrc)
print("{} directories for TOD.".format(len(dirs)))
print("{} files for calibration source.".format(len(fcalsrc)))
labels = []
for d in dirs:
    bla = str.split(d,'__')
    labels.append(bla[1])
# print(labels)

# print(len(fcalsrc))
# We want an array with all the starting times of the calsource data in unix time
times_calsrc = []
for namef in fcalsrc:
    timef = namef.split("_")[-1]
    times_calsrc.append("{}-{}-{}T{}:{}:{}".format(timef[0:4], timef[4:6], timef[6:8], timef[9:11], timef[11:13], timef[13:15]))
    # print(namef, times_calsrc[-1])
times_calsrc_astropy = Time(times_calsrc, format="isot", scale="utc")
times_calsrc_unix = np.array(times_calsrc_astropy.unix) - 0 * 3600#- 3600*2 # UTC + 2

### Some functions that will be useful

In [ ]:
def read_data_one_TES(thedir, AsicNum, TESNum):
    qfp = qubicfp()
    qfp.read_qubicstudio_dataset(thedir, asic=AsicNum)
    data = qfp.timeline(TES=TESNum)
    t_data = qfp.timeaxis(axistype='computertime', TES=TESNum)
    return t_data, data

In [ ]:
def demodulate_and_rebin(time, data, t_az, az, t_src, src, lowcut, highcut, fmod, nbins, elevation):
    ### Filter Data and Source Signal the same way
    FREQ_SAMPLING = 1./np.mean(time[1:] - time[:-1]) # not all time points are equally distributed
    filt = scsig.butter(5, [lowcut / FREQ_SAMPLING, highcut / FREQ_SAMPLING], btype='bandpass', output='sos')
    # Filter Data and change its sign to be in the same as Src
    new_data = -scsig.sosfilt(filt, data)
    # Interpolate Src on data times and filter it
    new_src = scsig.sosfilt(filt, np.interp(time, t_src, src))

    # Make the product for demodulation
    product = new_data * new_src / np.sum(new_src**2)

    # Smooth it over a period
    ppp = 1./fmod
    size_period = int(FREQ_SAMPLING * ppp)+1
    filter_period = np.ones((size_period,))/size_period
    mov_av = np.convolve(product, filter_period, mode='same') # I didn't check yet if the average shifts the results in az, but might not matter much?
    
    # Rebin this demodulated data as a function of azimuth corrected for elevation
    az_bin, amp_bin, daz, damp, others = ft.profile(np.interp(time, t_az, az),#*np.cos(np.radians(elevation)), 
                                              mov_av, nbins=nbins, cutbad=False,
                                              dispersion=True, plot=False, median=True)
    
    return az_bin, amp_bin, daz, damp



### Loop on all scans for one TES

In [ ]:
AsicNum = 1
TESNum = 93 #93#27
lowcut = 0.1
highcut = 15.
nscans = len(dirs)
nbins = 1000
freq_mod = 1 #Hz
# test
# delta_t = 2 * 3600 + 0.1 # there seems to be a shift between the two clocks of the order of 100ms (to be checked) + UTC+2
# delta_t = 2 * 3600 # there seems to be a shift between the two clocks of the order of 100ms (to be checked) + UTC+2
allscans_amp = np.zeros((nscans, nbins))
allscans_az = np.zeros((nscans, nbins))
allscans_el = np.zeros((nscans, nbins))
doplot = False
for idir, dir in enumerate(dirs): # we work scan by scan to avoid overflowing the memory
    # if idir not in [31]: # [31, 59, 95]
    #     doplot = True
    #     continue
    print(idir, dir)

    # read the mount data
    t_az, azinit = ft.read_hkintern(dir, thefieldname='Platform-Azimut')
    az = (azinit-2.**15)/(2.**16)*360
    elevation = float(dir[-5:])

    # read the scan data
    t_data, data = read_data_one_TES(dir, AsicNum, TESNum)

    # interpolate the mount data to time data
    az_data = np.interp(t_data, t_az, az)

    # read the calsource data
    ttsrc_i = []
    ddsrc_i = []
    for itime, time_cal in enumerate(times_calsrc_unix):
        # print(time_cal)
        if time_cal > np.max(t_data):
            print(time_cal, np.max(t_data))
            break
        try: # in case all the files are already read
            if times_calsrc_unix[itime + 1] > np.min(t_data): # we need only up to the file before the one that starts after the end of our acquisition
                data_calsrc = fitsio.read(fcalsrc[itime])
                thett = data_calsrc["timestamp"]
                thedd = data_calsrc["amplitude"]
                ttsrc_i.append(thett)# + delta_t)
                ddsrc_i.append(thedd)
        except IndexError: # it means we need the last one too (this is unlikely)
            print("We need up to the last calsource file. Check if there is an issue.")
            data_calsrc = fitsio.read(fcalsrc[itime])
            thett = data_calsrc["timestamp"]
            thedd = data_calsrc["amplitude"]
            ttsrc_i.append(thett)# + delta_t) # there seems to be a shift between the two clocks of the order of 100ms (to be checked)
            ddsrc_i.append(thedd)

    t_src = np.concatenate(ttsrc_i)
    data_src = np.concatenate(ddsrc_i)

    # test: it seems to be working!
    dt_data = np.min(t_az) - np.min(t_data) # ~3s
    dt_src = np.min(t_az) - np.min(t_src) # ~2h 0min 7s
    t_data += dt_data
    t_src += dt_src + 0.1 # there is still a slight shift

    # ang_bin, amp_bin, dang, damp = demodulate_and_rebin(t_data, data, t_az, az, t_src, data_src, 
    #                                                     lowcut, highcut, freq_mod, nbins, elevation)

    # data = data - np.median(data)
    az_bin, amp_bin, daz, damp = demodulate_and_rebin(t_data, data, t_az, az, t_src, data_src, 
                                                    lowcut, highcut, freq_mod, nbins, elevation)
    
    allscans_amp[idir] = amp_bin
    allscans_az[idir] = az_bin
    allscans_el[idir] = np.full(nbins, elevation)

    if doplot:
        fig, ax = plt.subplots()
        ax.set_title("At elevation={}".format(elevation))
        ax.plot(np.interp(t_data, t_az, az), data, label="raw data")
        ax1 = ax.twinx()
        ax1.plot(az_bin, amp_bin, label="demodulated and rebinned data")
        # ax.set_xlim(-12, -6)
        # ax.set_ylim(0.5e6, 1.5e6)
        plt.legend()
        plt.show()
        fig, ax = plt.subplots()
        ax.plot(t_data, (data-np.mean(data))/np.std(data),'.',label='TES')
        ax.plot(t_az, (az-np.mean(az))/np.std(az),'.',label='Az')
        # plt.plot(t_src, (data_src-np.mean(data_src))/np.std(data_src)/5,',',label='CalSrc')
        ax1 = ax.twinx()
        ax1.plot(t_src, (data_src - np.mean(data_src))/np.std(data_src), '.', c="r", label='CalSrc')
        ax1.set_ylim([-7, 4])
        plt.legend()
        # plt.ylim(-3,3)
        ax.set_ylim(-1.5, 2.5)
        # xlim = np.array([78, 86]) + 1.5555201e9
        # plt.xlim(xlim)
        plt.show()
        azr

In [ ]:
def healpix_map(azt, elt, tod, flags=None, flaglimit=0, nside=128, countcut=0, unseen_val=hp.UNSEEN):
    if flags is None:
        flags = np.zeros(len(azt))
    
    ok = flags <= flaglimit 
    return healpix_map_(azt[ok], elt[ok], tod[ok], nside=nside, countcut=countcut, unseen_val=unseen_val)


def healpix_map_(azt, elt, tod, nside=128, countcut=0, unseen_val=hp.UNSEEN):
# def healpix_map_(elt, azt, tod, nside=128, countcut=0, unseen_val=hp.UNSEEN):
    ips = hp.ang2pix(nside, azt, elt, lonlat=True)
    mymap = np.zeros(12*nside**2)
    mapcount = np.zeros(12*nside**2)
    for i in range(len(azt)):
        mymap[ips[i]] += tod[i]
        mapcount[ips[i]] += 1
    unseen = mapcount <= countcut
    mymap[unseen] = unseen_val
    mapcount[unseen] = unseen_val
    mymap[~unseen] = mymap[~unseen] / mapcount[~unseen]
    return mymap, mapcount

In [ ]:
# create a map in az el
map_1, map_count = healpix_map(allscans_az, allscans_el, allscans_amp, nside=340)

In [ ]:
az_source = allscans_az[65, 529] # 1.83521118 # degrees # by hand for now for TES 96

In [ ]:
def get_perp_vect(point_A, point_B, point_C): # get the vector perpendicular to the plane with these three points
    if len(np.shape(point_A)) == 2:
        vec_1 = (point_A - point_C[:, None])
    else:
        vec_1 = (point_A - point_C)
    if len(np.shape(point_B)) == 2:
        vec_2 = (point_B - point_C[:, None])
        if len(np.shape(point_A)) == 2:
            res_nonorm = np.cross(vec_1, vec_2, axis=0)
        else:
            res_nonorm = np.cross(vec_1[:, None], vec_2, axis=0)
    else:
        vec_2 = (point_B - point_C)
        if len(np.shape(point_A)) != 1:
            raise ValueError("Not the right shape for point_A")
        res_nonorm = np.cross(vec_1, vec_2, axis=0)
    print("shapes vec_1, vec_2, res_nonorm")
    print(np.shape(vec_1))
    print(np.shape(vec_2))
    print(np.shape(res_nonorm))
    return res_nonorm/np.linalg.norm(res_nonorm, axis=0)

def get_plane(point_A, point_B, point_C, offset=0): # plane equation ax + by + cz + d = 0
    perp_vect = get_perp_vect(point_A, point_B, point_C)
    a, b, c = perp_vect
    d = -np.dot(perp_vect, point_A)
    return np.array([a, b, c, d])

# get two orthogonal vectors on the plane, one pointing at point_ref_end from point_ref_start
def get_vect_plane(plane, point_ref_end, point_ref_start, offset=0): 
    if not np.isclose(np.dot(plane[:3], point_ref_end) + plane[3], 0):
        raise ValueError("The reference end point is not on the plane.")
    if not np.isclose(np.dot(plane[:3], point_ref_start) + plane[3], 0):
        raise ValueError("The reference start point is not on the plane.")
    perp_vect = plane[:3]/np.sqrt(np.sum(plane[:3]**2))
    vec_1 = (point_ref_end - point_ref_start)/np.linalg.norm(point_ref_end - point_ref_start)
    vec_2 = np.cross(perp_vect, vec_1)/np.linalg.norm(np.cross(vec_1, perp_vect))
    return vec_1, vec_2

def spherical2cartesian(rho, theta, phi, coord="spherical", axis="first"):
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.pi/2 - np.radians(phi.copy())
        phi = np.radians(theta_)
        # theta, phi = np.pi/2 - np.radians(phi), np.radians(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    x = rho * np.sin(theta) * np.cos(phi)
    y = rho * np.sin(theta) * np.sin(phi)
    z = rho * np.cos(theta)
    res = np.array([x, y, z])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    
def cartesian2spherical(x, y, z, coord="spherical", axis="first"):
    rho = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z/rho)
    phi = np.arctan2(y, x)
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.degrees(phi).copy()
        phi = 90 - np.degrees(theta_)
        # theta, phi = np.degrees(phi), 90 - np.degrees(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    res = np.array([rho, theta, phi])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    
def centre_coord_at_0(coord):
    return np.mod(coord - 180, 360) - 180


def mean_2d_bin_data(pos, values, bins): # pos and bins have shape (2, ...)
    # left limit of bins is excluded
    # shape of pos has to be (2, npoints), with npoints in the TOD

    # values_bini = jnp.digitize(pos[0], bins[0]) # right edge of each bin excluded, left edges included
    # values_binj = jnp.digitize(pos[1], bins[1])
    # res, _, _ = jnp.histogram2d(pos[0], pos[1], bins=bins, weights=values)
    res = histogram2d(pos[0], pos[1], range=((bins[0][0], bins[0][-1]), (bins[1][0], bins[1][-1])), bins=(len(bins[0]) - 1, len(bins[1]) - 1), weights=values)
    return res

In [ ]:
def match_shape(array1, shape_array2):
    return np.expand_dims(array1, list(np.arange(len(shape_array2))[1:]))

In [ ]:
# get the intersection between: the great circle perp to constant az = az_pointing and going though pointing and sphere centre ; the circle on the plane of constant az = az_source
# and then computes the delta elevation between the pointing and the intersection
def get_delta_el_distortion(azimuth_pointing, elevation_pointing, azimuth_calsource, sphere_centre=np.array([0, 0, 0]), sphere_radius=1): # à tester
    tilt = 1 # degrees # tilt to the pointing horizontal line, to account for a possible tilt of the beam # not perfectly implemented, just a test
    elevation_A = np.array([0])
    elevation_B = np.array([90])
    azimuth_calsource = np.array([azimuth_calsource]) # needs to be an array

    # we get vector perpendicular to the pointing great circle
    opposed_pointing = spherical2cartesian(sphere_radius, centre_coord_at_0(azimuth_pointing + 180 + tilt), 90 - elevation_pointing, coord="horizontal", axis="first")
    perp_vect_pointing = opposed_pointing - match_shape(sphere_centre, opposed_pointing.shape)

    # we get vector perpendicular to the meridian at source azimuth great circle
    point_A = spherical2cartesian(sphere_radius, azimuth_calsource, elevation_A, coord="horizontal", axis="first")
    point_B = spherical2cartesian(sphere_radius, azimuth_calsource, elevation_B, coord="horizontal", axis="first")
    perp_vec_merid_calsrc = get_perp_vect(point_A, point_B, sphere_centre) # vector perpendicular to meridian at calsource azimuth

    inter_vec = np.cross(perp_vect_pointing, perp_vec_merid_calsrc, axis=0)
    inter_vec /= np.linalg.norm(inter_vec, axis=0)
    inter_cart = inter_vec*sphere_radius + match_shape(sphere_centre, inter_vec.shape)
    x, y, z = inter_cart[0], inter_cart[1], inter_cart[2]
    return cartesian2spherical(x, y, z, coord="horizontal")



In [ ]:
intersection = get_delta_el_distortion(allscans_az, allscans_el, az_source, sphere_centre=np.array([0, 0, 0]), sphere_radius=1)
new_allscans_el = intersection[2]

In [ ]:
delta_el = allscans_el - intersection[2]
print(delta_el)

In [ ]:
def get_pix_edges(pix_min, pix_max, nb_pix):
    return np.linspace(pix_min, pix_max, nb_pix)
def get_pix_cent(pix_min, pix_max, nb_pix):
    pix_edges = get_pix_edges(pix_min, pix_max, nb_pix)
    return pix_edges + 0.5*(pix_max - pix_min)/nb_pix
pix_cent = [get_pix_cent(np.min(allscans_el), np.max(allscans_el), len(allscans_amp)), get_pix_cent(np.min(allscans_az), np.max(allscans_az), len(allscans_amp[0]))]
new_pix_cent = [get_pix_cent(np.min(new_allscans_el), np.max(new_allscans_el), len(allscans_amp)), get_pix_cent(np.min(allscans_az), np.max(allscans_az), len(allscans_amp[0]))]
original_map_interpolator = RegularGridInterpolator(pix_cent, allscans_amp, method='linear', bounds_error=False)
new_map = original_map_interpolator(np.moveaxis([new_allscans_el, allscans_az], 0, -1))[::-1]

In [ ]:
fig, ax = plt.subplots()
ax.imshow(-new_map, cmap=plt.cm.gray, vmin=-0.1, vmax=0.1, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
bins = [np.linspace(np.min(allscans_el), np.max(allscans_el), len(allscans_amp)), np.linspace(np.min(allscans_az), np.max(allscans_az), len(allscans_amp[0]))]
new_allscans_amp = mean_2d_bin_data([intersection[2], allscans_az], allscans_amp, bins)[::-1] # try replacing the original elevation with the new one

In [ ]:
fig, ax = plt.subplots()
ax.imshow(-new_allscans_amp, cmap=plt.cm.gray, vmin=-0.1, vmax=0.1, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.imshow(-allscans_amp, cmap=plt.cm.gray, vmin=-0.1, vmax=0.1, aspect=len(allscans_amp[0])/len(allscans_amp))
plt.show()

In [ ]:
fitsio.write("test_maps/TES_96.fits", -allscans_amp)
fitsio.write("test_maps/azimuth.fits", allscans_az)
fitsio.write("test_maps/elevation.fits", allscans_el)

In [ ]:
# check how the map looks
rot = (np.mean(allscans_az), np.mean(allscans_el))
map_1_flat = hp.gnomview(-map_1, rot=rot, xsize=1000, reso=2, return_projected_map=True, no_plot=True).data
map_1_flat[map_1_flat == hp.UNSEEN] = 0
plt.figure()
plt.imshow(map_1_flat, cmap=plt.cm.gray, vmin=-0.05, vmax=0.05)
plt.show()

In [ ]:
def spherical2cartesian(rho, theta, phi, coord="spherical", axis="first"):
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.pi/2 - np.radians(phi.copy())
        phi = np.radians(theta_)
        # theta, phi = np.pi/2 - np.radians(phi), np.radians(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    x = rho * np.sin(theta) * np.cos(phi)
    y = rho * np.sin(theta) * np.sin(phi)
    z = rho * np.cos(theta)
    res = np.array([x, y, z])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))
    

def get_simple_rotation_matrix(axis, angle):
    one = np.ones_like(angle)
    zero = np.zeros_like(angle)
    if axis == "x":
        R = np.array([[one, zero, zero],
                      [zero, np.cos(angle), -np.sin(angle)],
                      [zero, np.sin(angle), np.cos(angle)]])
    elif axis == "y":
        R = np.array([[np.cos(angle), zero, np.sin(angle)],
                      [zero, one, zero],
                      [-np.sin(angle), zero, np.cos(angle)]])
    elif axis == "z":
        R = np.array([[np.cos(angle), -np.sin(angle), zero],
                      [np.sin(angle), np.cos(angle), zero],
                      [zero, zero, one]])
    else:
        raise ValueError("{} is not a good axis.".format(axis))
    return R

def cartesian2spherical(x, y, z, coord="spherical", axis="first"):
    rho = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z/rho)
    phi = np.arctan2(y, x)
    if coord == "horizontal":
        theta_ = theta.copy()
        theta = np.degrees(phi).copy()
        phi = 90 - np.degrees(theta_)
        # theta, phi = np.degrees(phi), 90 - np.degrees(theta)
    elif coord != "spherical":
        raise ValueError("Argument coord = {} not understood.".format(coord))
    res = np.array([rho, theta, phi])
    if axis == "first":
        return res
    elif axis == "last":
        return np.moveaxis(res, 0, -1)
    else:
        raise ValueError("Argument axis = {} not understood.".format(axis))

In [ ]:
def get_new_azel(azt, elt, azsrc, elsrc): # adapted from get_new_azel_v2 in pipeline_moon_functions.py # 2 rotations to put the source at the right position?
    rho_sphere = 1
    moon_sph = np.radians(np.array([90 - elsrc, azsrc]))
    moon_pos = spherical2cartesian(rho_sphere, moon_sph[0], moon_sph[1])
    los_sph = np.radians(np.array([90 - elt, azt]))
    los_pos = spherical2cartesian(rho_sphere, los_sph[0], los_sph[1])

    ### simpler code
    rotation_matrix1 = get_simple_rotation_matrix("z", np.radians(azsrc))
    rotation_matrix2 = get_simple_rotation_matrix("y", np.radians(-elsrc))

    ### the last axis of both arrays becomes the first one in order to use np.dot
    # los_pos = np.moveaxis(los_pos, -1, 0)
    # rotation_matrix = np.moveaxis(rotation_matrix, -1, 0)
    # new_los_pos = np.dot(los_pos, rotation_matrix) # crashes
    # new_los_pos = np.vecmat(los_pos, rotation_matrix) # not yet in numpy 1.26...
    print("shape los_sph", np.shape(los_pos))
    # print("shape rotation_matrix", np.shape(rotation_matrix))
    # new_los_pos = np.einsum("il,ikl->kl", los_pos, rotation_matrix) # computation working!
    print("shape rotation_matrix", np.shape(rotation_matrix1), np.shape(rotation_matrix2))
    new_los_pos = np.einsum("il,ikl->kl", los_pos, rotation_matrix1)
    new_los_pos = np.einsum("il,ikl->kl", new_los_pos, rotation_matrix2)

    ### to see if we get always (0, 0, 0) for the Moon positions
    for i in range(3):
        print(moon_pos[i])
    new_moon_pos = np.einsum("il,ikl->kl", moon_pos, rotation_matrix1)
    print("First rotation")
    for i in range(3):
        print(new_moon_pos[i])
    new_moon_pos = np.einsum("il,ikl->kl", new_moon_pos, rotation_matrix2)
    print("Second rotation")
    for i in range(3):
        print(new_moon_pos[i])
    print(np.allclose(new_moon_pos[1], np.zeros_like(new_moon_pos[1])))
    print(np.allclose(new_moon_pos[2], np.zeros_like(new_moon_pos[2])))
    print(np.shape(new_los_pos))
    x, y, z = new_los_pos
    print(np.shape(x), np.shape(y), np.shape(z))
    res = cartesian2spherical(x, y, z)
    rho, theta, phi = res[..., 0], res[..., 1], res[..., 2]
    if not np.all(np.isclose(rho, 1)):
        raise ValueError("The radius of the sphere is not one.")
    
    newazt = np.degrees(phi)
    newelt = 90 - np.degrees(theta)
    return newazt, newelt
